# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [3]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [4]:
@function_tool
def lookup_evs_concept(concept: str) -> str:
    """Look up a concise explanation of a common environmental science concept."""
    concept_data = {
        "photosynthesis": (
            "Photosynthesis",
            "The process by which plants, algae, and some bacteria use sunlight to convert carbon dioxide and water into sugars, releasing oxygen.",
        ),
        "ozone depletion": (
            "Ozone depletion",
            "The thinning of the ozone layer in the stratosphere, mainly caused by ozone-depleting chemicals such as CFCs, which allows more harmful ultraviolet radiation to reach Earth.",
        ),
        "global warming": (
            "Global warming",
            "The long-term rise in Earth's average surface temperature, primarily driven by human greenhouse gas emissions.",
        ),
        "greenhouse effect": (
            "Greenhouse effect",
            "The warming that occurs when gases in Earth's atmosphere absorb and re-emit heat. The natural greenhouse effect keeps Earth habitable; human emissions are strengthening it.",
        ),
        "green house effect": (
            "Greenhouse effect",
            "The warming that occurs when gases in Earth's atmosphere absorb and re-emit heat. The natural greenhouse effect keeps Earth habitable; human emissions are strengthening it.",
        ),
    }

    concept_key = concept.strip().lower()
    if concept_key not in concept_data:
        return (
            f"I don't have information about {concept!r}. Try Photosynthesis, "
            "Ozone depletion, Global warming, or Greenhouse effect."
        )

    title, explanation = concept_data[concept_key]
    return f"{title}: {explanation}"


evs_agent = Agent(
    name="EVS Assistant",
    instructions="""
    You are a helpful Environmental Studies (EVS) assistant.
    Use lookup_evs_concept to answer questions about Photosynthesis, Ozone depletion,
    Global warming, or the Greenhouse effect. Give concise, student-friendly answers.
    """,
    tools=[lookup_evs_concept],
)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [ ]:
get_food_calories('banana')

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """)

In [ ]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)